# Figure 5

![Full Figure](../figures/fig05_FULL.png)

## Code for Figure 5B and 5D: model comparisons

**Goal:** Reproduce the submitted multinomial and binary model-comparison panels and their likelihood-ratio statistics.

| Panel | Content | Inputs |
|---|---|---|
| A | Multinomial history / interaction comparison | `scripts/fit_fig05.py` → `2025_07_22_fig5_compare_multi_intrxs_rs47` |
| B | Same models relative to the binary base | `scripts/fit_fig05.py` → `2025_07_22_fig5_compare_to_binary_rs47` |

Schematics are drawn separately. Why the binary-comparison improvement varies across animals is analyzed in [Figure 6](fig06.ipynb).

**Outputs:** `figures/fig05a_multinomial_comparison.png`, `figures/fig05b_binary_comparison.png`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from violations_glm.config import (
    BINARY_MODEL_COMPARE_COLOR,
    MODEL_COMPARE_COLOR,
    PREV_VIOLATION_COLOR,
)
from violations_glm.io import load_model_summary
from violations_glm.plotting import configure_publication_dpi, save_figure
from violations_glm.summaries import (
    bits_per_trial,
    geometric_mean_likelihood_ratio,
    overall_likelihood_ratio,
)

configure_publication_dpi()
sns.set_context("talk")

In [ ]:
MULTI = "2025_07_22_fig5_compare_multi_intrxs_rs47"
BINARY = "2025_07_22_fig5_compare_to_binary_rs47"

ORDER = [
    "base",
    "prev_viol_binary",
    "prev_viol_filtered",
    "prev_viol_binary_itrx",
    "prev_viol_filtered_itrx",
]
LABELS = [
    "Baseline",
    "+ Binary\nViolation History",
    "+ Filtered\nViolation History",
    "+ Binary\nViolation History\n& Stim Intrxs",
    "+ Filtered\nViolation History\n& Stim Intrxs",
]
TICK_COLORS = [
    "gray",
    PREV_VIOLATION_COLOR,
    "darkgreen",
    "orange",
    "#ce8641",
]

multi_fits = load_model_summary(MULTI)
multi_null = load_model_summary(MULTI, kind="null_models")
multi = bits_per_trial(multi_fits, multi_null).query("model_name != 'null'")

binary_fits = load_model_summary(BINARY)
binary_null = load_model_summary(BINARY, kind="null_models")
binary = bits_per_trial(binary_fits, binary_null).query("model_name != 'null'")

## Panel A — Multinomial model comparison

Fits are from [`scripts/fit_fig05.py`](../scripts/fit_fig05.py); this notebook loads stable summaries only.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.despine(ax=ax)
sns.pointplot(
    data=multi,
    x="model_name",
    y="bits_per_trial_relative_to_base",
    hue="animal_id",
    order=ORDER,
    palette=["gray"] * multi["animal_id"].nunique(),
    alpha=0.5,
    ax=ax,
)
sns.pointplot(
    data=multi,
    x="model_name",
    y="bits_per_trial_relative_to_base",
    order=ORDER,
    color=MODEL_COMPARE_COLOR,
    errorbar=None,
    ax=ax,
)
for index, model_name in enumerate(ORDER):
    if model_name == ORDER[0]:
        continue
    mean_delta = multi.loc[
        multi["model_name"] == model_name, "bits_per_trial_relative_to_base"
    ].mean()
    text = f"+{mean_delta:.4f}" if mean_delta > 0 else f"{mean_delta:.4f}"
    ax.text(index, 0.0005, text, ha="center", color=MODEL_COMPARE_COLOR)
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=45)
for tick, color in zip(ax.get_xticklabels(), TICK_COLORS, strict=True):
    tick.set_color(color)
ax.legend().remove()
ax.set(xlabel="", title="", ylabel=r"$\Delta$ test LL (bits per trial)")
fig.tight_layout()
save_figure(fig, "fig05a_multinomial_comparison")

In [ ]:
multi.groupby("model_name").bits_per_trial_relative_to_base.agg(
    ["mean", "std"])

## Base vs Intrx Filtered

In [ ]:
overall_likelihood_ratio(
    multi_fits,
    base_model_name="base",
    new_model_name="prev_viol_filtered_itrx",
)

In [ ]:
geometric_mean_likelihood_ratio(
    multi_fits,
    base_model_name="base",
    new_model_name="prev_viol_filtered_itrx",
)

> On average, each animal's test data is $10^{145}$ times more likely under the previous violation filtered model

## Filtered vs Intrx Filtered

In [ ]:
overall_likelihood_ratio(
    multi_fits,
    base_model_name="prev_viol_filtered",
    new_model_name="prev_viol_filtered_itrx",
)

In [ ]:
geometric_mean_likelihood_ratio(
    multi_fits,
    base_model_name="prev_viol_filtered",
    new_model_name="prev_viol_filtered_itrx",
)

> On average, each animal's test data is $10^{8}$ times more likely under the previous violation filtered model

## Panel B — Comparison with the binary base model

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.despine(ax=ax)
sns.pointplot(
    data=binary,
    x="model_name",
    y="bits_per_trial_relative_to_base",
    hue="animal_id",
    order=ORDER,
    palette=["gray"] * binary["animal_id"].nunique(),
    alpha=0.5,
    ax=ax,
)
ax.axhline(y=0, color="black")
sns.pointplot(
    data=binary,
    x="model_name",
    y="bits_per_trial_relative_to_base",
    order=ORDER,
    color=BINARY_MODEL_COMPARE_COLOR,
    errorbar=None,
    ax=ax,
)
for index, model_name in enumerate(ORDER):
    if model_name == ORDER[0]:
        continue
    mean_delta = binary.loc[
        binary["model_name"] == model_name, "bits_per_trial_relative_to_base"
    ].mean()
    text = f"+{mean_delta:.4f}" if mean_delta > 0 else f"{mean_delta:.4f}"
    ax.text(index, -0.007, text, ha="center", color=BINARY_MODEL_COMPARE_COLOR)
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=45)
for tick, color in zip(ax.get_xticklabels(), TICK_COLORS, strict=True):
    tick.set_color(color)
ax.legend().remove()
ax.set(
    xlabel="",
    title="",
    ylabel=r"$\Delta$ test LL (bits per trial)",
    ylim=(-0.008, 0.006),
)
fig.tight_layout()
save_figure(fig, "fig05b_binary_comparison")

In [ ]:
binary.groupby("model_name").bits_per_trial_relative_to_base.agg(
    ["mean", "std"])

## Base vs Intrx Filtered

In [ ]:
overall_likelihood_ratio(
    binary_fits,
    base_model_name="base",
    new_model_name="prev_viol_filtered_itrx",
)

In [ ]:
geometric_mean_likelihood_ratio(
    binary_fits,
    base_model_name="base",
    new_model_name="prev_viol_filtered_itrx",
)

> On average, each animal's test data is $10^{6}$ times more likely under the previous violation filtered model

## Filtered vs Intrx Filtered

In [ ]:
overall_likelihood_ratio(
    binary_fits,
    base_model_name="prev_viol_filtered",
    new_model_name="prev_viol_filtered_itrx",
)

In [ ]:
geometric_mean_likelihood_ratio(
    binary_fits,
    base_model_name="prev_viol_filtered",
    new_model_name="prev_viol_filtered_itrx",
)

> On average, each animal's test data is $10^{8}$ times more likely under the previous violation filtered model